# Fine-Tuning Qwen3-4B for SDG Classification (16-bit Precision)

**Model**: Qwen3-4B-Instruct  
**Task**: UN Sustainable Development Goals (SDG) classification with 3-step reasoning  
**Hardware**: Google Colab with A100 GPU  
**Precision**: 16-bit (bfloat16/float16) for maximum quality  
**Method**: LoRA fine-tuning with optimized parameters

---


## 1. Environment Setup & Installation

In [1]:
%%capture
# Install required packages
import subprocess
import sys

# Install unsloth and dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "unsloth"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers==4.56.2"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "trl==0.22.2"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "accelerate", "peft", "bitsandbytes"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "huggingface_hub"])

print(" All packages installed successfully!")

## 2. Mount Google Drive & Configure Paths

In [15]:
from google.colab import drive
import os

# Mount Google Drive
#drive.mount('/content/drive')

# Configure paths
DATA_FILE_PATH = "/content/split_1_final_classification_SFT_ChatML.jsonl"
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/AAI646/SFT_model/qwen3-4b-sdg-16bit"
LOCAL_OUTPUT_PATH = "/content/outputs"

# Create directories if they don't exist
os.makedirs(os.path.dirname(DRIVE_OUTPUT_PATH), exist_ok=True)
os.makedirs(LOCAL_OUTPUT_PATH, exist_ok=True)

print(f" Google Drive mounted successfully!")
print(f" Data file: {DATA_FILE_PATH}")
print(f" Model will be saved to: {DRIVE_OUTPUT_PATH}")

 Google Drive mounted successfully!
 Data file: /content/split_1_final_classification_SFT_ChatML.jsonl
 Model will be saved to: /content/drive/MyDrive/AAI646/SFT_model/qwen3-4b-sdg-16bit


## 3. Load Model in 16-bit Precision

Loading the model with **full 16-bit precision** (bfloat16) for maximum quality.

In [3]:
from unsloth import FastLanguageModel
import torch

# Model configuration
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 4096

print(f" Loading model: {MODEL_NAME}")
print(f"  Precision: 16-bit (bfloat16/float16)")
print(f" Max sequence length: {MAX_SEQ_LENGTH}")

# Load model in 16-bit precision
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,  # No quantization
    load_in_8bit=False,  # No quantization
    dtype=None,  # Auto-detect best dtype (bfloat16 on A100)
    device_map="auto",
)

print(f" Model loaded successfully!")
print(f" Precision: {model.dtype}")
print(f" VRAM usage: ~{torch.cuda.memory_allocated() / 1024**3:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
 Loading model: unsloth/Qwen3-4B-Instruct-2507
  Precision: 16-bit (bfloat16/float16)
 Max sequence length: 4096
==((====))==  Unsloth 2025.12.7: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

 Model loaded successfully!
 Precision: torch.bfloat16
 VRAM usage: ~7.52 GB


## 4. Configure LoRA Adapters

**LoRA Configuration**:
- **Rank (r)**: 64 - Higher rank for better learning capacity
- **Alpha**: 64 - Scaling factor for LoRA weights
- **Target modules**: All attention and MLP layers
- **Dropout**: 0.05 - Slight regularization to prevent overfitting
- **Gradient checkpointing**: Enabled for memory efficiency

In [4]:
# Add LoRA adapters with optimized parameters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more parameters = better quality
    lora_alpha=64,  # Scaling factor
    lora_dropout=0.05,  # Slight dropout for regularization
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",  # MLP layers
    ],
    use_gradient_checkpointing="unsloth",  # Memory efficient
    random_state=3407,
    use_rslora=False,  # Standard LoRA
)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = 100 * trainable_params / total_params

print(f" LoRA adapters configured!")
print(f" Trainable parameters: {trainable_params:,} ({trainable_percentage:.2f}%)")
print(f" Total parameters: {total_params:,}")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.12.7 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


 LoRA adapters configured!
 Trainable parameters: 132,120,576 (3.18%)
 Total parameters: 4,154,588,672


## 5. Load and Prepare Dataset

In [5]:
from datasets import load_dataset

# Load dataset
print(f" Loading dataset from: {DATA_FILE_PATH}")
dataset = load_dataset("json", data_files=DATA_FILE_PATH, split="train")

# Rename 'messages' to 'conversations' if needed (for compatibility)
if "messages" in dataset.column_names:
    dataset = dataset.rename_column("messages", "conversations")

print(f" Dataset loaded: {len(dataset)} examples")
print(f" Columns: {dataset.column_names}")

# Show a sample
print("\n Sample example:")
print(dataset[0]['conversations'])

 Loading dataset from: /content/split_1_final_classification_SFT_ChatML.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

 Dataset loaded: 1797 examples
 Columns: ['conversations']

 Sample example:
[{'role': 'system', 'content': 'You are an expert UN SDG Analyst. Analyze the text, generate a detailed 3-step reasoning process, and determine the final SDG classification.'}, {'role': 'user', 'content': 'â€¢ showcase impact stories emerging from Griffith University staff initiatives. Report scope and boundary Unless otherwise stated, this report covers the period 1 January 2023 to 31 December 2023 for Griffith University. Global Sustainability Frameworks and commitments United Nations Global Compact'}, {'role': 'assistant', 'content': 'Step 1: Let me Understand the paragraph\nThe text outlines elements of a report structure, including a bullet point about showcasing impact stories from university staff initiatives, a statement on the report\'s scope and boundary (covering Griffith University for the calendar year 2023), and a reference to the United Nations Global Compact as part of global sustainability fra

## 6. Apply Chat Template & Format Dataset

In [6]:
from unsloth.chat_templates import get_chat_template

# Apply Qwen3 chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-instruct",
)

print(" Chat template applied (Qwen3 format)")

# Format dataset using chat template
def formatting_prompts_func(examples):
    """Convert conversations to formatted text using chat template"""
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

print(" Dataset formatted with chat template")
print(f"\n Sample formatted text (first 500 chars):")
print(dataset[0]['text'][:500] + "...")

 Chat template applied (Qwen3 format)


Map:   0%|          | 0/1797 [00:00<?, ? examples/s]

 Dataset formatted with chat template

 Sample formatted text (first 500 chars):
<|im_start|>system
You are an expert UN SDG Analyst. Analyze the text, generate a detailed 3-step reasoning process, and determine the final SDG classification.<|im_end|>
<|im_start|>user
â€¢ showcase impact stories emerging from Griffith University staff initiatives. Report scope and boundary Unless otherwise stated, this report covers the period 1 January 2023 to 31 December 2023 for Griffith University. Global Sustainability Frameworks and commitments United Nations Global Compact<|im_end|>
<...


## 7. Configure Training Parameters

**Optimized Training Configuration for 16-bit Fine-tuning**:

### Learning Parameters:
- **Learning Rate**: 5e-5 (lower for stable 16-bit training)
- **Warmup Steps**: 100 (gradual learning rate increase)
- **LR Scheduler**: Cosine decay (smooth learning rate reduction)
- **Epochs**: 3 (sufficient for learning format and content)

### Batch Configuration (Optimized for A100):
- **Per Device Batch Size**: 4
- **Gradient Accumulation**: 4 steps
- **Effective Batch Size**: 16 (4 × 4)

### Optimization:
- **Optimizer**: AdamW (8-bit for memory efficiency)
- **Weight Decay**: 0.01 (regularization)
- **Max Grad Norm**: 1.0 (gradient clipping)
- **Precision**: bfloat16 (best for A100)

In [7]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# Training configuration optimized for quality
training_args = SFTConfig(
    # Output settings
    output_dir=LOCAL_OUTPUT_PATH,

    # Training schedule
    num_train_epochs=3,

    # Batch size (optimized for A100)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 16

    # Learning rate & scheduler
    learning_rate=5e-5,  # Lower LR for stable 16-bit training
    lr_scheduler_type="cosine",
    warmup_steps=100,

    # Optimization
    optim="adamw_8bit",  # Memory-efficient AdamW
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Precision (16-bit)
    bf16=True,  # Use bfloat16 on A100
    fp16=False,

    # Logging & saving
    logging_steps=10,
    save_steps=250,
    save_total_limit=3,

    # Other settings
    seed=3407,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
)

print(" Training configuration created")
print(f"\n Training Parameters:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Learning Rate: {training_args.learning_rate}")
print(f"   Batch Size: {training_args.per_device_train_batch_size}")
print(f"   Gradient Accumulation: {training_args.gradient_accumulation_steps}")
print(f"   Effective Batch Size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Precision: {'bfloat16' if training_args.bf16 else 'float16' if training_args.fp16 else 'float32'}")
print(f"   Optimizer: {training_args.optim}")

 Training configuration created

 Training Parameters:
   Epochs: 3
   Learning Rate: 5e-05
   Batch Size: 4
   Gradient Accumulation: 4
   Effective Batch Size: 16
   Precision: bfloat16
   Optimizer: OptimizerNames.ADAMW_8BIT


## 8. Initialize Trainer

In [8]:
from unsloth.chat_templates import train_on_responses_only

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

# Configure to train only on assistant responses (not system/user prompts)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("Trainer initialized")
print("Training only on assistant responses (not input prompts)")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1797 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=16):   0%|          | 0/1797 [00:00<?, ? examples/s]

Trainer initialized
Training only on assistant responses (not input prompts)


## 9. Check GPU Status

In [9]:
# Check GPU memory
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"  GPU: {gpu_stats.name}")
print(f" Memory Used: {start_gpu_memory} GB / {max_memory} GB")
print(f" Memory Available: {max_memory - start_gpu_memory:.2f} GB")

# Estimate training memory requirements
estimated_training_memory = start_gpu_memory + 10  # Rough estimate
print(f"\n Estimated peak memory during training: ~{estimated_training_memory:.1f} GB")

if estimated_training_memory > max_memory:
    print("  Warning: May run out of memory. Consider reducing batch size.")
else:
    print(" Sufficient memory for training!")

  GPU: NVIDIA A100-SXM4-80GB
 Memory Used: 8.018 GB / 79.318 GB
 Memory Available: 71.30 GB

 Estimated peak memory during training: ~18.0 GB
 Sufficient memory for training!


## 10. Start Training


In [10]:
# Start training
print("="*80)
print(" STARTING FINE-TUNING")
print("="*80)
print(f" Training on {len(dataset)} SDG classification examples")
print(f" Learning 3-step reasoning for UN SDG classification")
print(f"  Estimated time: ~{len(dataset) * 3 / (4 * 4) / 60:.1f} hours (A100)")
print("="*80)
print()

# Train the model
trainer_stats = trainer.train()

print("\n" + "="*80)
print(" TRAINING COMPLETED!")
print("="*80)
print(f" Final Loss: {trainer_stats.training_loss:.4f}")
print(f" Training Time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f" Samples/Second: {trainer_stats.metrics['train_samples_per_second']:.2f}")

 STARTING FINE-TUNING
 Training on 1797 SDG classification examples
 Learning 3-step reasoning for UN SDG classification
  Estimated time: ~5.6 hours (A100)



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,797 | Num Epochs = 3 | Total steps = 339
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.212500
20,2.038600
30,1.552500
40,1.229500
50,1.023300
60,0.852900
70,0.761400
80,0.719300
90,0.659200
100,0.653100



 TRAINING COMPLETED!
 Final Loss: 0.7268
 Training Time: 1017.58 seconds
 Samples/Second: 5.30


## 11. Test the Fine-tuned Model

In [29]:
from transformers import TextStreamer

# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Test message
test_messages = [
    {
        "role": "system",
        "content": "You are an expert UN SDG Analyst. Analyze the text, generate a detailed 3-step reasoning process, and determine the final SDG classification."
    },
    {
        "role": "user",
        "content": "For these countries, it is therefore difficult to conclude whether or not these developments have created a greater need for redistribution. A less ambiguous answer seems possible for the slight majority of countries in Figure 2, where preferences for inequality reduction have either strengthened or remained about the same. For most of these countries (Australia, Czech Republic, Germany, New Zealand, Norway, Poland, and Sweden), both inequality trends and changes in preferences consistently point towards a greater demand for redistribution. As shown in the discussion below, however, country patterns of actual observed changes in redistribution policies are not as clear-cut."
    }
]

# Format with chat template
prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

print("="*80)
print(" TESTING FINE-TUNED MODEL")
print("="*80)
print(f"\n Input: {test_messages[1]['content']}")
print("\n Model Response:\n")

# Generate response with streaming
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer, skip_prompt=True)

outputs = model.generate(
    **inputs,
    max_new_tokens=3500,
    temperature=0.3,  # Lower temperature for more consistent output
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.1,
    do_sample=True,
    streamer=streamer,
)

print("\n" + "="*80)
print(" Test completed!")
print("="*80)


 TESTING FINE-TUNED MODEL

 Input: For these countries, it is therefore difficult to conclude whether or not these developments have created a greater need for redistribution. A less ambiguous answer seems possible for the slight majority of countries in Figure 2, where preferences for inequality reduction have either strengthened or remained about the same. For most of these countries (Australia, Czech Republic, Germany, New Zealand, Norway, Poland, and Sweden), both inequality trends and changes in preferences consistently point towards a greater demand for redistribution. As shown in the discussion below, however, country patterns of actual observed changes in redistribution policies are not as clear-cut.

 Model Response:

Step 1: Let me Understand the paragraph
The text discusses challenges in assessing whether economic developments have increased the need for redistribution among certain countries, based on data from Figure 2. It highlights that for many countries, there's eviden

#SFT Model Testing for Classification Accuracy

In [28]:
import pandas as pd
import torch
from transformers import TextStreamer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report
import re
from tqdm import tqdm
from datetime import datetime

# Load your Excel file
print("="*80)
print("LOADING DATA")
print("="*80)
df = pd.read_excel('/content/SDG_SFT_Model_test_file.xlsx')  # Replace with your actual file path
print(f"✓ Loaded {len(df)} rows from Excel file")
print(f"✓ Columns: {df.columns.tolist()}")
print(f"✓ First few ground truth labels: {df['sdg'].head().tolist()}")

# Enable fast inference mode
FastLanguageModel.for_inference(model)
print("✓ Model ready for inference\n")

def extract_final_sdg(response_text):
    """Extract SDG classification from model response with robust parsing."""
    # Try multiple patterns in order of specificity

    # Pattern 1: "Final Classification: SDG1,SDG8" or "Final Classification: SDG1, SDG8"
    final_match = re.search(r'Final Classification:\s*(SDG\s*\d+(?:\s*,\s*SDG\s*\d+)*)', response_text, re.IGNORECASE)
    if final_match:
        return final_match.group(1).strip()

    # Pattern 2: "Conclusion: ... classified as SDG1,SDG8"
    conclusion_match = re.search(r'classified as\s*(SDG\s*\d+(?:\s*,\s*SDG\s*\d+)*)', response_text, re.IGNORECASE)
    if conclusion_match:
        return conclusion_match.group(1).strip()

    # Pattern 3: Look for NOSDG specifically
    if re.search(r'\bNOSDG\b', response_text, re.IGNORECASE):
        return "NOSDG"

    # Pattern 4: Find the last line that starts with "Final Classification" or similar
    lines = response_text.strip().split('\n')
    for line in reversed(lines):
        if 'final classification' in line.lower() or 'conclusion' in line.lower():
            sdg_match = re.findall(r'SDG\s*\d+', line, re.IGNORECASE)
            if sdg_match:
                return ', '.join(sdg_match)
            if 'NOSDG' in line.upper():
                return "NOSDG"

    # Pattern 5: Find all SDG mentions in the last paragraph
    paragraphs = response_text.strip().split('\n\n')
    if paragraphs:
        last_para = paragraphs[-1]
        sdg_matches = re.findall(r'SDG\s*\d+', last_para, re.IGNORECASE)
        if sdg_matches:
            return ', '.join(sdg_matches[-3:])  # Take last 3 mentions max

    # Pattern 6: Find ANY SDG pattern in the text (last resort)
    all_sdgs = re.findall(r'SDG\s*\d+', response_text, re.IGNORECASE)
    if all_sdgs:
        # Take the last few unique mentions
        unique_sdgs = []
        for sdg in reversed(all_sdgs):
            if sdg not in unique_sdgs:
                unique_sdgs.insert(0, sdg)
            if len(unique_sdgs) >= 3:
                break
        return ', '.join(unique_sdgs)

    return None

def normalize_sdg(sdg_text):
    """Normalize SDG labels to consistent format and handle NOSDG"""
    if pd.isna(sdg_text) or sdg_text is None or str(sdg_text).strip() == '':
        return set()

    # Convert to string and uppercase
    sdg_text = str(sdg_text).upper().replace(' ', '')

    # Check for NOSDG
    if 'NOSDG' in sdg_text:
        return {'NOSDG'}

    # Extract all SDG numbers
    sdg_numbers = re.findall(r'SDG(\d+)', sdg_text)

    if not sdg_numbers:
        return {'NOSDG'}  # Default to NOSDG if nothing found

    # Return as a set of normalized SDGs
    return {f'SDG{num}' for num in sdg_numbers}

def calculate_partial_credit(predicted_set, ground_truth_set):
    """
    Calculate partial credit where:
    - Predicting extra SDGs is OK (no penalty)
    - Missing SDGs gets penalized
    - Only matches count toward credit
    """
    if not ground_truth_set:
        ground_truth_set = {'NOSDG'}
    if not predicted_set:
        predicted_set = {'NOSDG'}

    # Special case: both are NOSDG
    if ground_truth_set == {'NOSDG'} and predicted_set == {'NOSDG'}:
        return 1.0, True, "Exact match (NOSDG)"

    # Special case: ground truth is NOSDG but prediction has SDGs
    if ground_truth_set == {'NOSDG'} and predicted_set != {'NOSDG'}:
        return 0.0, False, "Predicted SDGs when ground truth is NOSDG"

    # Special case: prediction is NOSDG but ground truth has SDGs
    if predicted_set == {'NOSDG'} and ground_truth_set != {'NOSDG'}:
        return 0.0, False, "Predicted NOSDG when ground truth has SDGs"

    # Calculate matches
    matches = predicted_set.intersection(ground_truth_set)
    missing = ground_truth_set - predicted_set
    extra = predicted_set - ground_truth_set

    # Partial credit = matches / ground_truth_count
    # Extra predictions don't penalize, only missing ones do
    if len(ground_truth_set) == 0:
        score = 0.0
    else:
        score = len(matches) / len(ground_truth_set)

    # Consider it "correct" if score >= 1.0 (all ground truth SDGs found, extras OK)
    is_correct = (score >= 1.0)

    details = f"Matches: {matches}, Missing: {missing}, Extra: {extra}"

    return score, is_correct, details

def predict_sdg(text, show_response=False):
    """Generate SDG prediction for a given text"""
    test_messages = [
        {
            "role": "system",
            "content": "You are an expert UN SDG Analyst. Analyze the text, generate a detailed 3-step reasoning process, and determine the final SDG classification."
        },
        {
            "role": "user",
            "content": text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        test_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    if show_response:
        streamer = TextStreamer(tokenizer, skip_special_tokens=True, skip_prompt=True)
        outputs = model.generate(
            **inputs,
            max_new_tokens=3500,
            temperature=0.3,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            do_sample=True,
            streamer=streamer,
        )
    else:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=3500,
                temperature=0.3,
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.1,
                do_sample=True,
            )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "assistant" in response.lower():
        response = response.split("assistant")[-1]

    return response

# Process all rows
predictions = []
ground_truths = []
partial_scores = []
failed_extractions = []
correct_count = 0
partial_count = 0
incorrect_count = 0
total_score = 0.0

print("="*80)
print("STARTING PREDICTIONS")
print("="*80)
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

for idx, row in df.iterrows():
    text = row['text']
    ground_truth_raw = row['sdg']
    ground_truth_set = normalize_sdg(ground_truth_raw)

    if pd.isna(text) or text.strip() == '':
        print(f"⚠️  Row {idx}: Skipping empty text")
        continue

    print(f"\n{'='*80}")
    print(f"SAMPLE {idx + 1}/{len(df)}")
    print(f"{'='*80}")
    print(f"Input Text: {text[:150]}{'...' if len(text) > 150 else ''}")
    print(f"Ground Truth: {ground_truth_raw} → {sorted(ground_truth_set)}")
    print(f"\nGenerating prediction...")

    # Show full response for first 3 examples
    show_full = (idx < 3)

    if show_full:
        print("\nModel Response:")
        print("-" * 80)

    response = predict_sdg(text, show_response=show_full)

    if show_full:
        print("-" * 80)

    predicted_sdg_raw = extract_final_sdg(response)
    predicted_set = normalize_sdg(predicted_sdg_raw)

    # Calculate partial credit
    score, is_correct, details = calculate_partial_credit(predicted_set, ground_truth_set)

    predictions.append(predicted_set)
    ground_truths.append(ground_truth_set)
    partial_scores.append(score)
    total_score += score

    if is_correct:
        correct_count += 1
        status = "✓ CORRECT (100%)"
        color = "🟢"
    elif score > 0:
        partial_count += 1
        status = f"◐ PARTIAL ({score*100:.0f}%)"
        color = "🟡"
    else:
        incorrect_count += 1
        status = "✗ INCORRECT (0%)"
        color = "🔴"

    print(f"\n{color} Predicted: {predicted_sdg_raw} → {sorted(predicted_set)}")
    print(f"   Status: {status}")
    print(f"   Details: {details}")

    if predicted_sdg_raw is None or not predicted_set:
        failed_extractions.append({
            'index': idx,
            'text': text[:100] + '...',
            'response': response[-500:]  # Last 500 chars
        })
        print(f"   ⚠️  Warning: Failed to extract SDG from response")

    # Running statistics
    total_processed = correct_count + partial_count + incorrect_count
    current_accuracy = correct_count / total_processed if total_processed > 0 else 0
    avg_score = total_score / total_processed if total_processed > 0 else 0

    print(f"\n📊 Running Stats:")
    print(f"   ✓ Fully Correct: {correct_count} ({correct_count/total_processed*100:.1f}%)")
    print(f"   ◐ Partial Credit: {partial_count} ({partial_count/total_processed*100:.1f}%)")
    print(f"   ✗ Incorrect: {incorrect_count} ({incorrect_count/total_processed*100:.1f}%)")
    print(f"   Average Score: {avg_score:.3f} ({avg_score*100:.1f}%)")

    # Progress bar
    progress = (idx + 1) / len(df) * 100
    bar_length = 40
    filled = int(bar_length * progress / 100)
    bar = '█' * filled + '░' * (bar_length - filled)
    print(f"   Progress: [{bar}] {progress:.1f}%")

# Calculate final metrics
print("\n" + "="*80)
print("FINAL EVALUATION RESULTS")
print("="*80)
print(f"Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

total_samples = len(predictions)
avg_partial_score = sum(partial_scores) / total_samples if total_samples > 0 else 0

print(f"📈 Overall Metrics:")
print(f"   Total Samples: {total_samples}")
print(f"   ✓ Fully Correct: {correct_count} ({correct_count/total_samples*100:.2f}%)")
print(f"   ◐ Partial Credit: {partial_count} ({partial_count/total_samples*100:.2f}%)")
print(f"   ✗ Incorrect: {incorrect_count} ({incorrect_count/total_samples*100:.2f}%)")
print(f"   Average Partial Credit Score: {avg_partial_score:.4f} ({avg_partial_score*100:.2f}%)")

# Exact match accuracy
exact_match_accuracy = correct_count / total_samples if total_samples > 0 else 0
print(f"\n📊 Exact Match Accuracy: {exact_match_accuracy:.4f} ({exact_match_accuracy*100:.2f}%)")

# Show failed extractions
if failed_extractions:
    print(f"\n⚠️  WARNING: Failed to extract SDG from {len(failed_extractions)} responses")
    print("\nFirst few examples:")
    for item in failed_extractions[:5]:
        print(f"\n  📍 Index {item['index']}:")
        print(f"     Text: {item['text']}")
        print(f"     Response end: {item['response']}")

# Create detailed results dataframe
results_df = pd.DataFrame({
    'text': df['text'][:len(predictions)],
    'ground_truth_raw': df['sdg'][:len(predictions)],
    'ground_truth_set': [sorted(list(gt)) for gt in ground_truths],
    'predicted_set': [sorted(list(pred)) for pred in predictions],
    'partial_score': partial_scores,
    'exact_match': [score >= 1.0 for score in partial_scores],
})

# Save results
output_file = 'sdg_predictions_results_with_partial_credit.xlsx'
results_df.to_excel(output_file, index=False)
print(f"\n💾 Detailed results saved to '{output_file}'")

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Total samples processed: {total_samples}")
print(f"✓ Exact matches: {correct_count} ({correct_count/total_samples*100:.1f}%)")
print(f"◐ Partial matches: {partial_count} ({partial_count/total_samples*100:.1f}%)")
print(f"✗ No matches: {incorrect_count} ({incorrect_count/total_samples*100:.1f}%)")
print(f"⚠️  Failed extractions: {len(failed_extractions)}")
print(f"📊 Average score: {avg_partial_score:.3f}")
print("="*80)
print("✅ EVALUATION COMPLETE!")
print("="*80)

LOADING DATA
✓ Loaded 29 rows from Excel file
✓ Columns: ['text', 'sdg']
✓ First few ground truth labels: ['NOSDG', 'NOSDG', 'SDG1, SDG8', 'SDG1', 'SDG10']
✓ Model ready for inference

STARTING PREDICTIONS
Started at: 2025-12-19 23:48:47


SAMPLE 1/29
Input Text: The net revaluation results are credited or debited to other comprehensive revenue and expense and are accumulated to the fi xed asset revaluation res...
Ground Truth: NOSDG → ['NOSDG']

Generating prediction...

Model Response:
--------------------------------------------------------------------------------
Step 1: Let me Understand the paragraph
The text describes accounting procedures for fixed assets under IFRS, specifically how changes in asset values (revaluations) are handled through reserves, equity adjustments, and surplus/deficit entries. It covers credit/debit treatments for revaluations, accumulation in reserves, handling of debit balances, reversal mechanisms, and recognition order when reversing prior decreases.


##SAVING MODEL TO GOOGLE DRIVE

In [16]:
import os

print("="*80)
print("💾 SAVING MODEL TO GOOGLE DRIVE")
print("="*80)
print(f"\n📁 Destination: {DRIVE_OUTPUT_PATH}")
print(f"⚙️  Format: 16-bit merged (bfloat16/float16)")
print(f"📦 Estimated size: ~8-9 GB")
print()

# Save merged 16-bit model
print("🔄 Saving merged 16-bit model...")
model.save_pretrained_merged(
    DRIVE_OUTPUT_PATH,
    tokenizer,
    save_method="merged_16bit",
)

print("\n✅ Model saved successfully to Google Drive!")
print(f"📂 Location: {DRIVE_OUTPUT_PATH}")
print("\n💡 You can load this model later with:")
print(f'   model, tokenizer = FastLanguageModel.from_pretrained("{DRIVE_OUTPUT_PATH}", load_in_4bit=True)')

💾 SAVING MODEL TO GOOGLE DRIVE

📁 Destination: /content/drive/MyDrive/AAI646/SFT_model/qwen3-4b-sdg-16bit
⚙️  Format: 16-bit merged (bfloat16/float16)
📦 Estimated size: ~8-9 GB

🔄 Saving merged 16-bit model...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `/content/drive/MyDrive/AAI646/SFT_model/qwen3-4b-sdg-16bit`: 100%|██████████| 2/2 [00:18<00:00,  9.37s/it]


Successfully copied all 2 files from cache to `/content/drive/MyDrive/AAI646/SFT_model/qwen3-4b-sdg-16bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:55<00:00, 27.93s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/AAI646/SFT_model/qwen3-4b-sdg-16bit`

✅ Model saved successfully to Google Drive!
📂 Location: /content/drive/MyDrive/AAI646/SFT_model/qwen3-4b-sdg-16bit

💡 You can load this model later with:
   model, tokenizer = FastLanguageModel.from_pretrained("/content/drive/MyDrive/AAI646/SFT_model/qwen3-4b-sdg-16bit", load_in_4bit=True)


In [18]:
# Make sure you're still in the same session with the trained model
# Re-save with error checking

print("🔄 Re-saving model to Google Drive...")

# Save merged 16-bit model
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/SDG_Models/qwen3-4b-sdg-16bit-fixed"

try:
    model.save_pretrained_merged(
        DRIVE_OUTPUT_PATH,
        tokenizer,
        save_method="merged_16bit",
    )
    print("✅ Model saved successfully!")

    # Verify the save by listing files
    import os
    files = os.listdir(DRIVE_OUTPUT_PATH)
    for f in files:
        size = os.path.getsize(os.path.join(DRIVE_OUTPUT_PATH, f)) / (1024**2)
        print(f"  {f}: {size:.2f} MB")

except Exception as e:
    print(f"❌ Error saving: {e}")

🔄 Re-saving model to Google Drive...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `/content/drive/MyDrive/SDG_Models/qwen3-4b-sdg-16bit-fixed`: 100%|██████████| 2/2 [00:25<00:00, 12.70s/it]


Successfully copied all 2 files from cache to `/content/drive/MyDrive/SDG_Models/qwen3-4b-sdg-16bit-fixed`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:47<00:00, 23.98s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/SDG_Models/qwen3-4b-sdg-16bit-fixed`
✅ Model saved successfully!
  chat_template.jinja: 0.00 MB
  tokenizer_config.json: 0.01 MB
  special_tokens_map.json: 0.00 MB
  added_tokens.json: 0.00 MB
  vocab.json: 2.65 MB
  merges.txt: 1.59 MB
  tokenizer.json: 10.89 MB
  config.json: 0.00 MB
  .cache: 0.00 MB
  model.safetensors.index.json: 0.03 MB
  model-00001-of-00002.safetensors: 4737.11 MB
  model-00002-of-00002.safetensors: 2935.19 MB


## 14. Save LoRA Adapters

In [19]:
# Save LoRA adapters only (lightweight)
LORA_PATH = DRIVE_OUTPUT_PATH + "-lora-adapters"

print(f"💾 Saving LoRA adapters to: {LORA_PATH}")
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)

print(f"✅ LoRA adapters saved! (~200-300 MB)")
print(f"📂 Location: {LORA_PATH}")
print("\n💡 Benefits of LoRA adapters:")
print("   - Smallest file size (~200 MB)")
print("   - Can merge with base model later")
print("   - Useful for version control")

💾 Saving LoRA adapters to: /content/drive/MyDrive/SDG_Models/qwen3-4b-sdg-16bit-fixed-lora-adapters
✅ LoRA adapters saved! (~200-300 MB)
📂 Location: /content/drive/MyDrive/SDG_Models/qwen3-4b-sdg-16bit-fixed-lora-adapters

💡 Benefits of LoRA adapters:
   - Smallest file size (~200 MB)
   - Can merge with base model later
   - Useful for version control


## 15. Final GPU Memory Report

In [20]:
# Final memory stats
final_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
memory_used = final_gpu_memory - start_gpu_memory

print("="*80)
print("📊 FINAL GPU MEMORY REPORT")
print("="*80)
print(f"GPU: {gpu_stats.name}")
print(f"Total Memory: {max_memory} GB")
print(f"Peak Memory Used: {final_gpu_memory} GB")
print(f"Memory for Training: {memory_used:.2f} GB")
print(f"Memory Efficiency: {(final_gpu_memory/max_memory)*100:.1f}%")
print("="*80)

📊 FINAL GPU MEMORY REPORT
GPU: NVIDIA A100-SXM4-80GB
Total Memory: 79.318 GB
Peak Memory Used: 12.41 GB
Memory for Training: 4.39 GB
Memory Efficiency: 15.6%


#Upload on hugging face

In [22]:
from huggingface_hub import HfApi, login

# Login to HuggingFace (get token from https://huggingface.co/settings/tokens)
login(token="hfiHkx")

# Push model to your HuggingFace account
print("⬆️ Uploading to HuggingFace Hub...")
model.push_to_hub_merged(
    "Tilakcsp/qwen3-4b-sdg-16bit",
    tokenizer,
    save_method="merged_16bit",
    token="hf_lfudDs",
    private=True,  # Keep it private
)
print("✅ Model uploaded to HuggingFace Hub!")
print("📁 Access at: https://huggingface.co/Tilakcsp/qwen3-4b-sdg-16bit")

⬆️ Uploading to HuggingFace Hub...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-sdg-16bit/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          | 4.22MB / 3.08GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:24<00:00, 102.02s/it]


Unsloth: Merge process complete. Saved to `/content/Tilakcsp/qwen3-4b-sdg-16bit`
✅ Model uploaded to HuggingFace Hub!
📁 Access at: https://huggingface.co/Tilakcsp/qwen3-4b-sdg-16bit


# Part 2 GSPO TRAINING

---

# Part 2: GSPO (Group Sequence Policy Optimization) Training

**Continuing from SFT to GSPO for enhanced learning**

After the supervised fine-tuning (SFT), we'll apply GRPO optimization where:
- The model generates **4 samples** per input text
- Each sample is compared with ground truth labels
- The model receives rewards for correct predictions
- This reinforcement learning approach helps the model learn from its own generations

**Key Features:**
- Sequence-level importance sampling (GSPO mode)
- Sophisticated reward function with partial credit for correct labels
- No penalty for extra predictions, only for missing labels
- Format and reasoning bonuses

## 16. Install GSPO Dependencies

In [30]:
%%capture
# Install/upgrade TRL for GSPO support
import subprocess
import sys

# Upgrade TRL to latest version with GSPO support
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "trl>=0.15.0"])

print("✅ GRPO dependencies installed!")

## 17. Define Reward Functions for GSPO

The reward system:
- **Partial Credit**: +0.5 for each correctly predicted SDG label
- **Perfect Match Bonus**: +0.5 if ALL ground truth labels are found
- **False Negative Penalty**: -0.4 for each missed label
- **No penalty** for predicting extra labels (false positives)
- **Format Bonus**: +0.1 for proper 3-step reasoning structure
- **Reasoning Bonus**: +0.1 for quality reasoning indicators:

In [31]:
import re
from typing import Set, List

# ============================================================================
# REWARD FUNCTION DESIGN - CRITICAL FOR GSPO
# ============================================================================

def extract_sdg_labels_from_response(response: str) -> Set[str]:
    """
    Extract SDG labels from the model's response.
    Looks for patterns like 'SDG10', 'SDG1', 'NOSDG' in the final classification.
    """
    # Look for "Final Classification:" section
    final_section = response.split("Final Classification:")[-1] if "Final Classification:" in response else response

    # Extract all SDG mentions (SDG1-SDG17 or NOSDG)
    sdg_pattern = r'\b(SDG\d{1,2}|NOSDG)\b'
    found_labels = set(re.findall(sdg_pattern, final_section, re.IGNORECASE))

    # Normalize to uppercase
    found_labels = {label.upper() for label in found_labels}

    return found_labels


def parse_ground_truth_labels(label_string: str) -> Set[str]:
    """
    Parse ground truth labels from format like:
    'SDG1', 'SDG2, SDG16', 'SDG12, SDG17, SDG14', 'NOSDG'
    """
    if not label_string or label_string == "":
        return set()

    # Split by comma and clean
    labels = [label.strip().upper() for label in str(label_string).split(',')]
    return set(labels)


def compute_sdg_reward(
    response: str,
    ground_truth: str,
    correct_label_reward: float = 0.5,
    perfect_match_bonus: float = 0.5,
    false_negative_penalty: float = -0.4,
    format_bonus: float = 0.1,
    reasoning_bonus: float = 0.1
) -> float:
    """
    Sophisticated reward function for SDG classification.

    Reward Components:
    - Partial credit: +0.5 for each correct label predicted
    - Perfect match bonus: +0.5 if ALL ground truth labels found
    - False negative penalty: -0.4 for each missed label
    - No penalty for false positives (extra predictions)
    - Format bonus: +0.1 for proper structure
    - Reasoning bonus: +0.1 for quality reasoning

    Examples:
    - Ground truth: SDG2, SDG16 | Predicted: SDG2, SDG16
      → 0.5 + 0.5 + 0.5 = 1.5 (both correct + perfect match)

    - Ground truth: SDG2, SDG16 | Predicted: SDG2, SDG16, SDG10
      → 0.5 + 0.5 + 0.5 = 1.5 (both correct + perfect match, no penalty for SDG10)

    - Ground truth: SDG2, SDG16 | Predicted: SDG2
      → 0.5 - 0.4 = 0.1 (one correct, one missed)
    """
    predicted = extract_sdg_labels_from_response(response)
    ground_truth_set = parse_ground_truth_labels(ground_truth)

    # If ground truth is empty, treat as error
    if not ground_truth_set:
        return -1.0

    # Calculate set operations
    correct_predictions = predicted & ground_truth_set  # Intersection
    false_negatives = ground_truth_set - predicted  # What we missed

    reward = 0.0

    # 1. Partial credit for EACH correct label
    reward += len(correct_predictions) * correct_label_reward

    # 2. Perfect match bonus - all ground truth labels were found
    if len(false_negatives) == 0 and len(ground_truth_set) > 0:
        reward += perfect_match_bonus

    # 3. Penalty ONLY for missing labels (false negatives)
    reward += len(false_negatives) * false_negative_penalty

    # 4. Format bonus - check if response follows expected structure
    has_steps = "Step 1:" in response and "Step 2:" in response and "Step 3:" in response
    has_conclusion = "Conclusion:" in response or "Final Classification:" in response

    if has_steps and has_conclusion:
        reward += format_bonus

    # 5. Reasoning bonus - check for quality reasoning indicators
    reasoning_keywords = ["because", "analyzes", "discusses", "relates to", "focuses on", "aligned with"]
    reasoning_count = sum(1 for keyword in reasoning_keywords if keyword.lower() in response.lower())

    if reasoning_count >= 3:
        reward += reasoning_bonus

    return reward


# Test the reward function
print("🧪 Testing reward function...")
test_response = """Step 1: Let me understand...
Step 2: The possible SDGs are...
Step 3: Final analysis...
Conclusion: The text relates to poverty reduction.
Final Classification: SDG1, SDG2"""

test_gt = "SDG1, SDG2"
reward = compute_sdg_reward(test_response, test_gt)
print(f"Test response predicted: {extract_sdg_labels_from_response(test_response)}")
print(f"Ground truth: {parse_ground_truth_labels(test_gt)}")
print(f"Reward: {reward}")
print("✅ Reward functions defined!")

🧪 Testing reward function...
Test response predicted: {'SDG2', 'SDG1'}
Ground truth: {'SDG2', 'SDG1'}
Reward: 1.6
✅ Reward functions defined!


## 18. Prepare Dataset for GSPO Training

**Configuration Options:**
- `GRPO_DATA_PATH` - Path to your JSONL dataset file
- `GRPO_USE_SAMPLING` - Set `True` to use random sampling, `False` for full dataset
- `GRPO_SAMPLE_SIZE` - Number of random samples (e.g., 60)

Convert the SFT dataset to GRPO format:
- Extract `prompt` (user text with system message)
- Extract `ground_truth` (SDG labels from assistant response)

In [32]:
import json
import random
from datasets import Dataset

# ============================================================================
# GSPO DATASET CONFIGURATION
# ============================================================================

# Path to your JSONL dataset file (ChatML format)
GRPO_DATA_PATH = "/content/split_2_final_classification_SFT_ChatML.jsonl"  # <-- Set your dataset path here

# Sampling configuration
GRPO_USE_SAMPLING = True   # Set to True to use random sampling, False to use full dataset
GRPO_SAMPLE_SIZE = 60      # Number of random samples to use (only if GRPO_USE_SAMPLING=True)

# Set random seed for reproducibility
GRPO_RANDOM_SEED = 42

print("=" * 60)
print("📊 GRPO DATASET CONFIGURATION")
print("=" * 60)
print(f"📂 Dataset Path: {GRPO_DATA_PATH}")
print(f"🎲 Use Sampling: {GRPO_USE_SAMPLING}")
if GRPO_USE_SAMPLING:
    print(f"📏 Sample Size: {GRPO_SAMPLE_SIZE}")
print(f"🌱 Random Seed: {GRPO_RANDOM_SEED}")
print("=" * 60)

# ============================================================================

def extract_ground_truth_from_assistant(assistant_content: str) -> str:
    """Extract SDG labels from the assistant's response."""
    # Look for "Final Classification:" line
    if "Final Classification:" in assistant_content:
        final_line = assistant_content.split("Final Classification:")[-1].strip()
        # Extract SDG labels from this line
        sdg_pattern = r'\b(SDG\d{1,2}|NOSDG)\b'
        labels = re.findall(sdg_pattern, final_line, re.IGNORECASE)
        return ", ".join([l.upper() for l in labels])
    return ""

def prepare_grpo_dataset(jsonl_path: str) -> Dataset:
    """
    Convert ChatML JSONL to GRPO format.

    GRPO needs:
    - prompt: The formatted prompt (will be used to generate responses)
    - ground_truth: The correct SDG labels
    """
    data = []

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            messages = item.get('messages', [])

            # Extract components
            system_content = ""
            user_content = ""
            assistant_content = ""

            for msg in messages:
                if msg['role'] == 'system':
                    system_content = msg['content']
                elif msg['role'] == 'user':
                    user_content = msg['content']
                elif msg['role'] == 'assistant':
                    assistant_content = msg['content']

            # Extract ground truth SDG labels
            ground_truth = extract_ground_truth_from_assistant(assistant_content)

            if ground_truth and user_content:
                # Format prompt using chat template (system + user)
                prompt_messages = [
                    {"role": "system", "content": system_content},
                    {"role": "user", "content": user_content}
                ]

                # Apply chat template
                formatted_prompt = tokenizer.apply_chat_template(
                    prompt_messages,
                    tokenize=False,
                    add_generation_prompt=True
                )

                data.append({
                    'prompt': formatted_prompt,
                    'ground_truth': ground_truth,
                    'text': user_content  # Original text for reference
                })

    return Dataset.from_list(data)

# Prepare GRPO dataset
print(f"\n📂 Loading data from: {GRPO_DATA_PATH}")
grpo_dataset_full = prepare_grpo_dataset(GRPO_DATA_PATH)
print(f"📊 Full dataset size: {len(grpo_dataset_full)} examples")

# Apply sampling if enabled
if GRPO_USE_SAMPLING:
    random.seed(GRPO_RANDOM_SEED)
    sample_size = min(GRPO_SAMPLE_SIZE, len(grpo_dataset_full))
    sampled_indices = random.sample(range(len(grpo_dataset_full)), sample_size)
    grpo_dataset = grpo_dataset_full.select(sampled_indices)
    print(f"🎲 Sampled {sample_size} random examples for GRPO training")
else:
    grpo_dataset = grpo_dataset_full
    print(f"📊 Using full dataset for GRPO training")

print(f"\n✅ GRPO dataset prepared: {len(grpo_dataset)} examples")
print(f"📊 Columns: {grpo_dataset.column_names}")

# Show sample
print("\n📝 Sample entry:")
print(f"Ground Truth: {grpo_dataset[0]['ground_truth']}")
print(f"Text (first 200 chars): {grpo_dataset[0]['text'][:200]}...")


📊 GRPO DATASET CONFIGURATION
📂 Dataset Path: /content/split_2_final_classification_SFT_ChatML.jsonl
🎲 Use Sampling: True
📏 Sample Size: 60
🌱 Random Seed: 42

📂 Loading data from: /content/split_2_final_classification_SFT_ChatML.jsonl
📊 Full dataset size: 1797 examples
🎲 Sampled 60 random examples for GRPO training

✅ GRPO dataset prepared: 60 examples
📊 Columns: ['prompt', 'ground_truth', 'text']

📝 Sample entry:
Ground Truth: SDG5, SDG16
Text (first 200 chars): Rebecca Kadaga, first woman Speaker of the Ugandan Parliament, also agrees that there is a need to focus beyond increasing the number of women in government to increasing their effectiveness in politi...


## 19. Configure GSPO Trainer

**GSPO Configuration:**
- **num_generations**: 4 (generate 4 responses per prompt)
- **Importance sampling**: Sequence-level (GSPO mode)
- **Learning rate**: 1e-6 (very small for RL fine-tuning)
- **KL penalty**: 0.01 (prevent model from deviating too much from base)

In [36]:
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig

# GRPO Output directory
GRPO_OUTPUT_PATH = LOCAL_OUTPUT_PATH + "/grpo"
GRPO_DRIVE_PATH = DRIVE_OUTPUT_PATH + "-grpo"

# Create reward function wrapper for GRPO
def sdg_reward_function(completions: List[str], prompts: List[str] = None, **kwargs) -> List[float]:
    """
    Reward function for GRPOTrainer.

    Args:
        completions: List of model-generated completions
        prompts: List of prompts (optional, for reference)
        **kwargs: May contain 'ground_truth' from dataset

    Returns:
        List of reward scores
    """
    rewards = []

    # Get ground truths - they should be passed via the dataset metadata
    ground_truths = kwargs.get('ground_truth', [None] * len(completions))

    for i, completion in enumerate(completions):
        gt = ground_truths[i] if i < len(ground_truths) else None

        if gt:
            reward = compute_sdg_reward(completion, gt)
        else:
            # If no ground truth, give neutral reward based on format only
            has_format = "Step 1:" in completion and "Final Classification:" in completion
            reward = 0.2 if has_format else 0.0

        rewards.append(reward)

    return rewards

# GSPO Configuration
grpo_config = GRPOConfig(
    output_dir=GRPO_OUTPUT_PATH,

    # Training schedule
    max_steps=150,  # Number of optimization steps

    # Batch configuration
    per_device_train_batch_size=1,  # Keep small due to memory
    gradient_accumulation_steps=4,

    # GRPO-specific parameters
    num_generations=4,  # Generate 4 responses per prompt
    max_completion_length=1500,  # Max tokens to generate

    # Learning rate (very small for RL)
    learning_rate=1e-6,

    # KL penalty to prevent model divergence
    beta=0.01,

    # Generation parameters
    temperature=0.7,

    # Optimization
    max_grad_norm=1.0,

    # Logging
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none",

    # Precision
    bf16=True,
)

# LoRA config for GRPO (if not already using LoRA from SFT)
grpo_lora_config = LoraConfig(
    r=64,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

print(" GRPO configuration created!")
print(f"\n GRPO Parameters:")
print(f"   Max Steps: {grpo_config.max_steps}")
print(f"   Generations per prompt: {grpo_config.num_generations}")
print(f"   Learning Rate: {grpo_config.learning_rate}")
print(f"   KL Penalty (beta): {grpo_config.beta}")
print(f"   Max Completion Length: {grpo_config.max_completion_length}")
print(f"   Temperature: {grpo_config.temperature}")

 GRPO configuration created!

 GRPO Parameters:
   Max Steps: 150
   Generations per prompt: 4
   Learning Rate: 1e-06
   KL Penalty (beta): 0.01
   Max Completion Length: 1500
   Temperature: 0.7


## 20. Initialize GSPO Trainer with Custom Reward Function

In [37]:
# Prepare the model for training (disable inference mode if enabled)
# Note: If you're continuing from SFT, the model is already loaded with LoRA

# Make sure model is in training mode
model.train()

# Store ground truth mapping for reward computation
ground_truth_map = {item['prompt']: item['ground_truth'] for item in grpo_dataset}

# Custom reward function that uses the ground truth mapping
def reward_function_with_gt(completions: List[str], prompts: List[str], **kwargs) -> List[float]:
    """
    Compute rewards by looking up ground truth from prompts.
    """
    rewards = []

    for prompt, completion in zip(prompts, completions):
        # Look up ground truth for this prompt
        gt = ground_truth_map.get(prompt, "")

        if gt:
            reward = compute_sdg_reward(completion, gt)
        else:
            # Fallback: reward based on format only
            has_format = "Step 1:" in completion and "Final Classification:" in completion
            reward = 0.2 if has_format else 0.0

        rewards.append(reward)

    return rewards

# Initialize GRPO Trainer
print("🚀 Initializing GRPO Trainer...")

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    reward_funcs=reward_function_with_gt,
    peft_config=None,  # Already has LoRA from SFT, or use grpo_lora_config for fresh
)

print("✅ GRPO Trainer initialized!")
print(f"📊 Training dataset size: {len(grpo_dataset)}")
print(f"🔄 Generations per prompt: {grpo_config.num_generations}")
print(f"📈 Total generations per step: {grpo_config.num_generations}")

🚀 Initializing GRPO Trainer...
✅ GRPO Trainer initialized!
📊 Training dataset size: 60
🔄 Generations per prompt: 4
📈 Total generations per step: 4


## 21. Start GSPO Training

The model will:
1. Generate 4 different responses for each input text
2. Compare each response with ground truth labels
3. Compute rewards based on correct/missing predictions
4. Update model weights to favor higher-reward generations

In [38]:
# Check GPU memory before training
print("=" * 80)
print("🚀 STARTING GRPO TRAINING")
print("=" * 80)

gpu_mem_before = torch.cuda.memory_allocated() / 1024**3
print(f"💾 GPU Memory before training: {gpu_mem_before:.2f} GB")
print(f"📊 Training on {len(grpo_dataset)} examples")
print(f"🔄 Generating {grpo_config.num_generations} samples per prompt")
print(f"📈 Max steps: {grpo_config.max_steps}")
print("=" * 80)
print()

# Start GRPO training
grpo_stats = grpo_trainer.train()

# Training complete
print("\n" + "=" * 80)
print("✅ GRPO TRAINING COMPLETED!")
print("=" * 80)

gpu_mem_after = torch.cuda.max_memory_allocated() / 1024**3
print(f"💾 Peak GPU Memory: {gpu_mem_after:.2f} GB")
print(f"⏱️ Training Time: {grpo_stats.metrics.get('train_runtime', 0):.2f} seconds")

🚀 STARTING GRPO TRAINING
💾 GPU Memory before training: 8.99 GB
📊 Training on 60 examples
🔄 Generating 4 samples per prompt
📈 Max steps: 150



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60 | Num Epochs = 3 | Total steps = 150
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)
/usr/local/lib/python3.12/dist-packages/unsloth/kernels/utils.py:963: UserWarning: An output with one or more elements was resized since it had shape [1, 4, 2560], which does not match the required output shape [4, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:31.)
  out = torch_matmul(X, W.t(), out = out)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_function_with_gt / mean,rewards / reward_function_with_gt / std
10,3297.724600,0.930000,0.087338,503.775000,476.400000,532.700000,0.000000,503.775000,476.400000,532.700000,329772.347876,0.930000,0.087338
20,0.084600,1.222500,0.155829,476.525000,442.900000,510.900000,0.000000,476.525000,442.900000,510.900000,8.459020,1.222500,0.155829
30,0.021300,0.835000,0.446448,539.925000,497.700000,595.400000,0.000000,539.925000,497.700000,595.400000,2.129423,0.835000,0.446448
40,0.023400,0.882500,0.028094,494.700000,467.200000,525.600000,0.000000,494.700000,467.200000,525.600000,2.340917,0.882500,0.028094
50,0.023000,0.960000,0.194392,511.100000,478.700000,543.600000,0.000000,511.100000,478.700000,543.600000,2.304362,0.960000,0.194392
60,0.022400,0.932500,0.140774,512.275000,479.700000,550.500000,0.000000,512.275000,479.700000,550.500000,2.240496,0.932500,0.140774
70,0.022100,1.297500,0.100774,500.750000,465.400000,546.000000,0.000000,500.750000,465.400000,546.000000,2.208043,1.297500,0.100774
80,0.022600,1.060000,0.384265,506.625000,469.000000,543.300000,0.000000,506.625000,469.000000,543.300000,2.255918,1.060000,0.384265
90,0.021700,0.930000,0.432665,525.475000,485.000000,574.800000,0.000000,525.475000,485.000000,574.800000,2.172661,0.930000,0.432665
100,0.022400,0.827500,0.075000,505.300000,480.000000,544.400000,0.000000,505.300000,480.000000,544.400000,2.239502,0.827500,0.075000


/usr/local/lib/python3.12/dist-packages/unsloth/kernels/utils.py:963: UserWarning: An output with one or more elements was resized since it had shape [1, 4, 2560], which does not match the required output shape [4, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:31.)
  out = torch_matmul(X, W.t(), out = out)



✅ GRPO TRAINING COMPLETED!
💾 Peak GPU Memory: 12.14 GB
⏱️ Training Time: 5921.74 seconds


## 23. Test GSPO-Optimized Model

Generate multiple samples to see the model's improved consistency.

In [40]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def test_grpo_model(text: str, num_samples: int = 4, temperature: float = 0.2):
    """
    Generate multiple samples from the GRPO-optimized model.

    Args:
        text: Input text to classify
        num_samples: Number of samples to generate
        temperature: Sampling temperature
    """
    messages = [
        {
            "role": "system",
            "content": "You are an expert UN SDG Analyst. Analyze the text, generate a detailed 3-step reasoning process, and determine the final SDG classification."
        },
        {
            "role": "user",
            "content": text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print("=" * 80)
    print(f"📝 Input Text: {text[:200]}...")
    print("=" * 80)

    all_predictions = []

    for i in range(num_samples):
        print(f"\n🔄 Generation {i+1}/{num_samples}")
        print("-" * 40)

        outputs = model.generate(
            **inputs,
            max_new_tokens=1500,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract assistant response
        if "<|im_start|>assistant" in response:
            response = response.split("<|im_start|>assistant")[-1].strip()

        # Extract predicted labels
        predicted_labels = extract_sdg_labels_from_response(response)
        all_predictions.append(predicted_labels)

        print(f"🏷️ Predicted SDGs: {', '.join(sorted(predicted_labels)) if predicted_labels else 'None'}")

    # Analyze consistency
    print("\n" + "=" * 80)
    print("📊 GENERATION CONSISTENCY ANALYSIS")
    print("=" * 80)

    # Count label frequencies
    from collections import Counter
    all_labels = []
    for pred in all_predictions:
        all_labels.extend(pred)

    label_counts = Counter(all_labels)

    print(f"\n📈 Label Frequencies across {num_samples} generations:")
    for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
        percentage = (count / num_samples) * 100
        print(f"   {label}: {count}/{num_samples} ({percentage:.0f}%)")

    # Check if all predictions are identical
    if len(set(frozenset(p) for p in all_predictions)) == 1:
        print(f"\n✅ PERFECT CONSISTENCY: All {num_samples} generations produced identical predictions!")
    else:
        unique_predictions = len(set(frozenset(p) for p in all_predictions))
        print(f"\n⚠️ {unique_predictions} unique prediction sets across {num_samples} generations")

    return all_predictions

# Test with sample texts
test_texts = [
    "The university installed solar panels on campus buildings to reduce carbon emissions and promote renewable energy use among students.",
    "For these countries, it is therefore difficult to conclude whether or not these developments have created a greater need for redistribution.",
    "The government launched a new healthcare initiative to provide free vaccinations to children in rural areas."
]

print(" TESTING GRPO-OPTIMIZED MODEL")
print("=" * 80)

for i, text in enumerate(test_texts):
    print(f"\n\n{'='*80}")
    print(f"TEST {i+1}")
    print(f"{'='*80}")
    test_grpo_model(text, num_samples=4, temperature=0.7)

 TESTING GRPO-OPTIMIZED MODEL


TEST 1
📝 Input Text: The university installed solar panels on campus buildings to reduce carbon emissions and promote renewable energy use among students....

🔄 Generation 1/4
----------------------------------------
🏷️ Predicted SDGs: SDG13, SDG7

🔄 Generation 2/4
----------------------------------------
🏷️ Predicted SDGs: SDG13, SDG7

🔄 Generation 3/4
----------------------------------------
🏷️ Predicted SDGs: SDG13, SDG7, SDG9

🔄 Generation 4/4
----------------------------------------
🏷️ Predicted SDGs: SDG13, SDG7

📊 GENERATION CONSISTENCY ANALYSIS

📈 Label Frequencies across 4 generations:
   SDG7: 4/4 (100%)
   SDG13: 4/4 (100%)
   SDG9: 1/4 (25%)

⚠️ 2 unique prediction sets across 4 generations


TEST 2
📝 Input Text: For these countries, it is therefore difficult to conclude whether or not these developments have created a greater need for redistribution....

🔄 Generation 1/4
----------------------------------------
🏷️ Predicted SDGs: NOSDG

🔄 Ge

## 24. Evaluate GSPO Model on Validation Set

Quick evaluation on a subset of training data to measure improvement.

In [45]:
import random
from tqdm import tqdm

def evaluate_model(dataset, model, tokenizer, num_samples=50, temperature=0.3):
    """
    Evaluate model on a subset of data.

    Returns metrics:
    - Exact match accuracy
    - Recall (% of ground truth labels found)
    - Average reward score
    """
    # Sample subset
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))

    exact_matches = 0
    total_recall = 0
    total_reward = 0

    results = []

    print(f"📊 Evaluating on {len(indices)} samples...")

    for idx in tqdm(indices, desc="Evaluating"):
        item = dataset[idx]
        prompt = item['prompt']
        ground_truth = item['ground_truth']

        # Generate prediction
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=1500,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract predictions
        predicted = extract_sdg_labels_from_response(response)
        gt_set = parse_ground_truth_labels(ground_truth)

        # Calculate metrics
        is_exact_match = predicted == gt_set
        recall = len(predicted & gt_set) / len(gt_set) if gt_set else 0
        reward = compute_sdg_reward(response, ground_truth)

        exact_matches += int(is_exact_match)
        total_recall += recall
        total_reward += reward

        results.append({
            'ground_truth': gt_set,
            'predicted': predicted,
            'exact_match': is_exact_match,
            'recall': recall,
            'reward': reward
        })

    # Calculate averages
    avg_accuracy = exact_matches / len(indices)
    avg_recall = total_recall / len(indices)
    avg_reward = total_reward / len(indices)

    return {
        'accuracy': avg_accuracy,
        'recall': avg_recall,
        'avg_reward': avg_reward,
        'results': results
    }

# Run evaluation
print("=" * 80)
print("📈 EVALUATING GSPO-OPTIMIZED MODEL")
print("=" * 80)

eval_results = evaluate_model(grpo_dataset, model, tokenizer, num_samples=20)

print("\n" + "=" * 80)
print("📊 EVALUATION RESULTS")
print("=" * 80)
print(f"✅ Exact Match Accuracy: {eval_results['accuracy']*100:.1f}%")
print(f"📈 Average Recall: {eval_results['recall']*100:.1f}%")
print(f"🎯 Average Reward Score: {eval_results['avg_reward']:.3f}")
print("=" * 80)

📈 EVALUATING GSPO-OPTIMIZED MODEL
📊 Evaluating on 20 samples...


Evaluating: 100%|██████████| 20/20 [11:22<00:00, 34.12s/it]


📊 EVALUATION RESULTS
✅ Exact Match Accuracy: 40.0%
📈 Average Recall: 82.5%
🎯 Average Reward Score: 0.980


##Single Sample test - Output

In [46]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def test_grpo_model(text: str):
    """
    Generate a single sample from the GRPO-optimized model.

    Args:
        text: Input text to classify
    """
    messages = [
        {
            "role": "system",
            "content": "You are an expert UN SDG Analyst. Analyze the text, generate a detailed 3-step reasoning process, and determine the final SDG classification."
        },
        {
            "role": "user",
            "content": text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print("=" * 80)
    print(f" Input Text: {text}")
    print("=" * 80)

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=1500,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract assistant response
    if "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1].strip()

    print("\n Model Response:")
    print("-" * 80)
    print(response)
    print("-" * 80)

    # Extract predicted labels
    predicted_labels = extract_sdg_labels_from_response(response)

    print(f"\n Predicted SDGs: {', '.join(sorted(predicted_labels)) if predicted_labels else 'None'}")
    print("=" * 80)

    return predicted_labels, response


# Test with sample text
test_text = "The university installed solar panels on campus buildings to reduce carbon emissions and promote renewable energy use among students."

print("\n TESTING GRPO-OPTIMIZED MODEL")
print("=" * 80)

predicted_labels, full_response = test_grpo_model(test_text)


 TESTING GRPO-OPTIMIZED MODEL
 Input Text: The university installed solar panels on campus buildings to reduce carbon emissions and promote renewable energy use among students.

 Model Response:
--------------------------------------------------------------------------------
system
You are an expert UN SDG Analyst. Analyze the text, generate a detailed 3-step reasoning process, and determine the final SDG classification.
user
The university installed solar panels on campus buildings to reduce carbon emissions and promote renewable energy use among students.
assistant
Step 1: Let me Understand the paragraph
The text describes a university initiative to install solar panels on campus buildings, with the stated purposes of reducing carbon emissions and promoting renewable energy use among students. The core message is an environmental action taken by an educational institution to lower greenhouse gas emissions and encourage sustainable energy practices on campus.

Step 2: Based on the an

## 26. Upload GSPO Model to HuggingFace Hub

In [41]:
from huggingface_hub import HfApi, login

HF_TOKEN = "hf_lTuQ"

# Repository name for GRPO model
HF_REPO_NAME = "Tilakcsp/qwen3-4b-sdg-gspo"

# Make it private or public
HF_PRIVATE = True  # Set to False for public

# ============================================================================

# Login to HuggingFace
print("🔐 Logging in to HuggingFace...")
login(token=HF_TOKEN)

# Push GRPO model to HuggingFace Hub
print("=" * 60)
print("⬆️ UPLOADING GRPO MODEL TO HUGGINGFACE HUB")
print("=" * 60)
print(f"📦 Repository: {HF_REPO_NAME}")
print(f"🔒 Private: {HF_PRIVATE}")
print()

model.push_to_hub_merged(
    HF_REPO_NAME,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
    private=HF_PRIVATE,
)

print("\n" + "=" * 60)
print("✅ GRPO MODEL UPLOADED TO HUGGINGFACE HUB!")
print("=" * 60)
print(f"📁 Access at: https://huggingface.co/{HF_REPO_NAME}")
print("=" * 60)


🔐 Logging in to HuggingFace...
⬆️ UPLOADING GRPO MODEL TO HUGGINGFACE HUB
📦 Repository: Tilakcsp/qwen3-4b-sdg-gspo
🔒 Private: True



Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...b-sdg-gspo/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          | 4.22MB / 3.08GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:42<00:00, 81.02s/it]


Unsloth: Merge process complete. Saved to `/content/Tilakcsp/qwen3-4b-sdg-gspo`

✅ GRPO MODEL UPLOADED TO HUGGINGFACE HUB!
📁 Access at: https://huggingface.co/Tilakcsp/qwen3-4b-sdg-gspo
